In [0]:
%run ./logger

### Step 1: Import Libraries

In [0]:
from pyspark.sql.functions import *
from datetime import datetime

In [0]:
batch_id = dbutils.widgets.get("batch_id")

if not batch_id:
    batch_id = "Batch_slver_001"

In [0]:
# batch_id = "Batch_slver_001"


start_time = log_start(
    batch_id,
    "silver_data",
    "bronze",
    "silver"
)

### Step 2: Create Database

### Step 3: Create Execution Log Table

In [0]:
spark.conf.set(
    "fs.azure.account.key.insurancedatahomecredit.blob.core.windows.net",
    "mvIlwSRtmT+AXedlxE40hoBJZsEQlvxnlHDD1ZdWje1R7m00TeP/QXcK1Gr7Nu8os0Mru1pMnb+E+AStH5S9mA=="
)

database,tableName,isTemporary
medallion_db,execution_logs,false


### Step 4: Start Logger Function

### Step 5: Success Logger Function

### Step 6: Failure Logger Function

### Step 7: View Execution Logs

Batch_ID,Notebook_Name,Source_Layer,Target_Layer,Start_Time,End_Time,Duration_Seconds,Status,Records_Read,Records_Written,Error_Message
12345,bronze_data,raw,bronze,2026-06-27T14:04:23.259962Z,2026-06-27T14:04:46.96848Z,23,SUCCESS,1018324,1017209,
12345,raw_data,raw,bronze,2026-06-27T14:03:45.092331Z,2026-06-27T14:04:09.163528Z,24,SUCCESS,1018324,1018324,


### read the file

In [0]:
bronze_df = spark.read \
    .format("delta") \
    .load("wasbs://bronze@insurancedatahomecredit.blob.core.windows.net/train")

In [0]:
silver_temp = bronze_df.dropDuplicates()

In [0]:
silver_temp = silver_temp.fillna({
    "Sales": 0,
    "Customers": 0,
    "Open": 0,
    "Promo": 0
})

silver_temp = silver_temp.withColumn(
    "StateHoliday",
    upper(col("StateHoliday"))
)

silver_temp = silver_temp.withColumn(
    "Date",
    to_date(col("Date"), "yyyy-MM-dd")
)

silver_temp = silver_temp.withColumn(
    "Sales",
    when(col("Sales") < 0, 0).otherwise(col("Sales"))
)

In [0]:
reject_df = silver_temp.filter(
    (col("Store").isNull()) |
    (col("Date").isNull())
)

In [0]:
silver_df = silver_temp.filter(
    col("Store").isNotNull() &
    col("Date").isNotNull()
)

### Remove Duplicate Records

### Step 3: Handle Null Values

### Standardize Text Columns

In [0]:
# from pyspark.sql.functions import upper,col

# silver_df = silver_df.withColumn(
#     "StateHoliday",
#     upper(col("StateHoliday"))
# )

### Standardize Date Format

In [0]:
# from pyspark.sql.functions import to_date

# silver_df = silver_df.withColumn(
#     "Date",
#     to_date(col("Date"),"yyyy-MM-dd")
# )

### Apply Business Rule

In [0]:
# from pyspark.sql.functions import when

# silver_df = silver_df.withColumn(
#     "Sales",
#     when(col("Sales") < 0,0)
#     .otherwise(col("Sales"))
# )

### Step 7: Create Reject Records

### Step 8: Keep Only Valid Records

In [0]:
# silver_df = silver_df.filter(
#     col("Store").isNotNull() &
#     col("Date").isNotNull()
# )

### Write Silver Delta

In [0]:
silver_df.write \
.format("delta") \
.mode("overwrite") \
.save("wasbs://silver@insurancedatahomecredit.blob.core.windows.net/train")


### Write Reject Records

In [0]:
reject_df.write \
.format("delta") \
.mode("overwrite") \
.save("wasbs://silver@insurancedatahomecredit.blob.core.windows.net/reject_records")

### Verify Silver Data

### Record Count Validation

In [0]:
records_read = bronze_df.count()
records_written = silver_df.count()


In [0]:
log_success(
    batch_id,
    "silver_data",
    "bronze",
    "silver",
    start_time,
    records_read,
    records_written
)